In [ ]:
from dotenv import load_dotenv
import os
import glob
from pathlib import Path
import gradio as gr
from openai import OpenAI

In [ ]:
load_dotenv(override=True)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY:
    print(f"OpenAI API Key exists and begins {OPENAI_API_KEY[:8]}")
else:
    print("OpenAI API Key not set")

In [ ]:
MODEL = "gpt-4.1-nano"
openai = OpenAI()

In [ ]:
knowledge = {}

filenames = glob.glob("knowledge-base/employees/*")

for filename in filenames:
    name = Path(filename).stem.split(' ')[-1]
    with open(filename, "r", encoding="utf8") as f:
        knowledge[name.lower()] = f.read()

In [ ]:
knowledge.get("kim")

In [ ]:
SYSTEM_PREFIX = """
You represent Insurellm, the Insurance Tech company.
You are an expert in answering questions about Insurellm; its employees and its products.
You are provided with additional context that might be relevant to the user's question.
Give brief, accurate answers. If you don't know the answer, say so.

Relevant context:
"""

In [ ]:
import re
from collections.abc import Mapping

def get_relevant_context_pyhthonic(message: str, knowledge: Mapping[str, str]) -> list[str]:
    """Return knowledge entries whose keys appear as words in `message`.

    Matching is case-insensitive; each entry is returned at most once,
    in order of first appearance.
    """
    words = re.findall(r"[a-z]+", message.lower())
    unique_words = dict.fromkeys(words)          # ordered de-dupe
    return [knowledge[w] for w in unique_words if w in knowledge]

In [ ]:
get_relevant_context_pyhthonic("Who is lancaster?", knowledge)

In [ ]:
def additional_context(message, knowledge):
    relevant_context = get_relevant_context_pyhthonic(message, knowledge)
    if not relevant_context:
        result = "There is no additional context relevant to the user's question."
    else:
        result = "The following additional context might be relevant in answering the user's question:\n\n"
        result += "\n\n".join(relevant_context)
    return result

In [ ]:
print(additional_context("Who is Alex Lancaster?", knowledge))

In [ ]:
def chat(message, history):
    system_message = SYSTEM_PREFIX + additional_context(message)
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content

In [ ]:
view = gr.ChatInterface(chat).launch(inbrowser=True)